# EuroSAT EfficientNetB0 Kaggle Runner

This notebook runs Phase 5 EfficientNetB0 Stage 1 and Stage 2 on Kaggle.
It assumes dataset input is mounted under `/kaggle/input` and all outputs are written to `/kaggle/working`.

In [ ]:
import shutil
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO = WORKING / 'satellite-land-classification-dl'

if REPO.exists():
    shutil.rmtree(REPO)

In [ ]:
!git clone https://github.com/milanovicmilos/satellite-land-classification-dl.git /kaggle/working/satellite-land-classification-dl
%cd /kaggle/working/satellite-land-classification-dl
!git checkout feature/efficientnet

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e .

## Dataset Path Setup

Update DATASET_INPUT_ROOT if your Kaggle dataset mount path is different.
Expected folder structure is .../EuroSAT/<class_name>/*.jpg.

In [ ]:
import json
from pathlib import Path

DATASET_INPUT_ROOT = Path('/kaggle/input/eurosat-rgb/EuroSAT')
assert DATASET_INPUT_ROOT.exists(), f'Missing dataset path: {DATASET_INPUT_ROOT}'

def patch_dataset_root(config_path: Path) -> None:
    payload = json.loads(config_path.read_text(encoding='utf-8'))
    payload['dataset_root'] = DATASET_INPUT_ROOT.as_posix()
    config_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')

patch_dataset_root(Path('configs/efficientnet_b0.stage1.json'))
patch_dataset_root(Path('configs/efficientnet_b0.stage2.json'))
print('Patched dataset_root to', DATASET_INPUT_ROOT)

In [ ]:
!python run.py --prepare-dataset --config configs/efficientnet_b0.stage1.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits

## Stage 1 Smoke (Frozen Backbone)

In [ ]:
!python run.py --run-baseline --config configs/efficientnet_b0.stage1.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits --reports-output /kaggle/working/artifacts/reports/efficientnet_b0_stage1_smoke.json --checkpoints-output /kaggle/working/checkpoints/efficientnet_b0/stage1

## Stage 2 Smoke (Unfrozen Backbone, Resume from Stage 1)

Stage 2 config points to checkpoints/efficientnet_b0/stage1/best_checkpoint.pt,
which resolves under /kaggle/working when running from repository root.

In [ ]:
!python run.py --run-baseline --config configs/efficientnet_b0.stage2.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits --reports-output /kaggle/working/artifacts/reports/efficientnet_b0_stage2_smoke.json --checkpoints-output /kaggle/working/checkpoints/efficientnet_b0/stage2

## Full Runs

After smoke validation, increase epochs in stage configs and re-run Stage 1 then Stage 2.